In [6]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np, torch
import sft

model = sft.load_ckpt("big_2026-08-16_06-45-06.pt")
packed = sft.build_or_load()
model.train()
T = model.cfg.block_size
print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M params, block {T}")


loaded big_2026-08-16_06-45-06.pt: step 19999, val_loss 1.3784
27.3M params, block 512


In [2]:
def grad_vector(B, rng):
    x, y, keep = sft._windows(packed, "train", B, T, rng)
    y = y.masked_fill(~keep, -100)
    model.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        _, loss = model(x, y)
    loss.backward()
    return torch.cat([
        (p.grad if p.grad is not None else torch.zeros_like(p)).flatten()
        for p in model.parameters()
    ])

In [3]:
N = 128
rng = np.random.default_rng(0)
sum_g, sum_sq = None, 0.0

for _ in range(N):
    g = grad_vector(1, rng)
    sum_g = g.clone() if sum_g is None else sum_g + g
    sum_sq += g.pow(2).sum().item()

mean_g = sum_g / N # type: ignore
m2   = sum_sq / N                    # -> ||g_true||^2 + sigma^2
mbar = mean_g.pow(2).sum().item()    # -> ||g_true||^2 + sigma^2/N

sigma2  = (m2 - mbar) * N / (N - 1)  # Bessel-style correction
g_true2 = mbar - sigma2 / N

print(f"mean ||g_i||^2   {m2:.4f}")
print(f"||g_bar_N||^2    {mbar:.4f}")
print(f"sigma^2          {sigma2:.4f}")
print(f"||g_true||^2     {g_true2:.4f}   ->  ||g_true|| = {g_true2**0.5:.4f}")


mean ||g_i||^2   36.6182
||g_bar_N||^2    5.0056
sigma^2          31.8616
||g_true||^2     4.7567   ->  ||g_true|| = 2.1810


In [4]:
print(f"{'B':>4} {'predicted':>10} {'measured':>10}")
for B in (1, 2, 4, 8):
    pred = (g_true2 + sigma2 / B) ** 0.5
    meas = np.mean([grad_vector(B, rng).norm().item() for _ in range(8)])
    print(f"{B:>4} {pred:>10.3f} {meas:>10.3f}")


   B  predicted   measured
   1      6.051      6.268
   2      4.548      4.511
   4      3.567      3.636
   8      2.956      2.854


In [5]:
for B in (16, 32, 64):
    print(f"B={B:>3}  predicted gnorm {(g_true2 + sigma2/B)**0.5:.3f}")


B= 16  predicted gnorm 2.598
B= 32  predicted gnorm 2.398
B= 64  predicted gnorm 2.292


In [6]:
for B in (1, 8, 16, 32, 64, 256):
    frac = (sigma2/B) / (g_true2 + sigma2/B)
    print(f"B={B:>4}  noise share of ||g||^2: {frac*100:5.1f}%")


B=   1  noise share of ||g||^2:  87.0%
B=   8  noise share of ||g||^2:  45.6%
B=  16  noise share of ||g||^2:  29.5%
B=  32  noise share of ||g||^2:  17.3%
B=  64  noise share of ||g||^2:   9.5%
B= 256  noise share of ||g||^2:   2.5%


Good — this is measurable in a few minutes. One caution first.

**Your 20k run is on the GPU right now** (~4.07 GB of 8.19). This demo needs ~1.4 GB at the batch sizes below, which fits in what's left, but don't raise `B` past 8 in step 4 or you risk OOM-killing a 75-minute job.

Step 1 — setup
--------------

Loading the **base** checkpoint deliberately: your run logged `gnorm 2.31` at step 0 with those exact weights at B=32, so we have a ground-truth number to check the estimate against at the end.

    import numpy as np, torch
    import sft
    
    model = sft.load_ckpt("big_2026-08-16_06-45-06.pt")
    packed = sft.build_or_load()
    model.train()
    T = model.cfg.block_size
    print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M params, block {T}")
    

`dropout=0.0` in `big_cfg`, so `model.train()` adds no randomness — every bit of variance we measure comes from data sampling, which is what σ² is defined as.

Step 2 — one gradient as a flat vector
--------------------------------------

    def grad_vector(B, rng):
        x, y, keep = sft._windows(packed, "train", B, T, rng)
        y = y.masked_fill(~keep, -100)
        model.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            _, loss = model(x, y)
        loss.backward()
        return torch.cat([
            (p.grad if p.grad is not None else torch.zeros_like(p)).flatten()
            for p in model.parameters()
        ])
    

Each call returns a 27.3M-element fp32 vector (109 MB), so never hold more than a couple at once.

Step 3 — estimate σ² and ‖g\_true‖²
-----------------------------------

We can't store 128 gradient vectors (14 GB), so accumulate two running quantities instead. The two estimators:

    mean‖g_i‖²  →  ‖g_true‖² + σ²        (single example: full noise)
    ‖ḡ_N‖²      →  ‖g_true‖² + σ²/N      (average of N: noise cut by N)
    

Subtract to isolate σ², then back out the signal:

    N = 128
    rng = np.random.default_rng(0)
    sum_g, sum_sq = None, 0.0
    
    for _ in range(N):
        g = grad_vector(1, rng)
        sum_g = g.clone() if sum_g is None else sum_g + g
        sum_sq += g.pow(2).sum().item()
    
    mean_g = sum_g / N
    m2   = sum_sq / N                    # -> ||g_true||^2 + sigma^2
    mbar = mean_g.pow(2).sum().item()    # -> ||g_true||^2 + sigma^2/N
    
    sigma2  = (m2 - mbar) * N / (N - 1)  # Bessel-style correction
    g_true2 = mbar - sigma2 / N
    
    print(f"mean ||g_i||^2   {m2:.4f}")
    print(f"||g_bar_N||^2    {mbar:.4f}")
    print(f"sigma^2          {sigma2:.4f}")
    print(f"||g_true||^2     {g_true2:.4f}   ->  ||g_true|| = {g_true2**0.5:.4f}")
    

Step 4 — check the 1/B law empirically
--------------------------------------

    print(f"{'B':>4} {'predicted':>10} {'measured':>10}")
    for B in (1, 2, 4, 8):
        pred = (g_true2 + sigma2 / B) ** 0.5
        meas = np.mean([grad_vector(B, rng).norm().item() for _ in range(8)])
        print(f"{B:>4} {pred:>10.3f} {meas:>10.3f}")
    

If the decomposition holds, those two columns track closely. This is the actual test — everything else is bookkeeping.

Step 5 — extrapolate and validate against your run
--------------------------------------------------

    for B in (16, 32, 64):
        print(f"B={B:>3}  predicted gnorm {(g_true2 + sigma2/B)**0.5:.3f}")
    

**B=32 should land near 2.31** — the step-0 gnorm your current run logged with these weights. B=16 should land near 2.65, the step-0 value from the earlier B=16 runs. That closes the loop against real logged numbers rather than self-consistency.

Step 6 — the noise share
------------------------

    for B in (1, 8, 16, 32, 64, 256):
        frac = (sigma2/B) / (g_true2 + sigma2/B)
        print(f"B={B:>4}  noise share of ||g||^2: {frac*100:5.1f}%")
    

Then free the memory
--------------------

    del sum_g, mean_g, model
    import gc; gc.collect(); torch.cuda.empty_cache()
    

**One prediction, so you can judge the result honestly:** at step 0 the base model is far from the instruct optimum, so ‖g\_true‖ is large and the noise share will be _much lower_ than the ~90% I estimated earlier. The check is the run's own logs — 2.65 at B=16 vs 2.31 at B=32 is a ratio of 1.15, nowhere near the 1.41 that pure noise would give. My 90% figure came from _converged_ gnorms (1.20 vs 0.89, ratio 0.74 ≈ 1/√2), which is a different regime.

That contrast is the real lesson: the noise share isn't a constant, it grows as the model converges and ‖g\_true‖ shrinks toward zero. If you want to see it directly, rerun steps 3–6 with `sft.load_ckpt(sft.latest_ckpt("sft"))` once your 20k run finishes.